# Video med OpenCV

Videoer i mappen `bilder/` ligger i Python-mappen, akkurat som bildene, så `cv2.VideoCapture("baller.mp4")` virker uten sti. OpenCV i nettleseren har FFmpeg innebygd og leser

- MP4, MOV, MKV og WebM med H.264, H.265, VP8 eller VP9 (det de fleste mobiler og kameraer lager)
- AVI med MJPEG eller MPEG-4 (Xvid)

`baller.mp4` er 6 sekunder med tre baller som spretter: en rød, en grønn og en blå.

Kjør cellene med **Shift+Enter**, eller trykk **Kjør alle**.

In [ ]:
import cv2
import numpy as np
from ipywidgets import interact

cap = cv2.VideoCapture("baller.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
antall = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
bredde = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
hoyde = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{bredde} × {hoyde} piksler, {fps:g} bilder per sekund, {antall} bilder ({antall / fps:.1f} s)")

ok, bilde = cap.read()        # det første bildet, som BGR
print(bilde.shape, bilde.dtype)
cv2.imshow("første bilde", bilde)

## Bla gjennom videoen

En video er bare mange bilder etter hverandre. `cap.set(cv2.CAP_PROP_POS_FRAMES, n)` hopper til bilde nummer `n`, og `cap.read()` gir det. Dra i glidebryteren.

In [ ]:
@interact(n=(0, antall - 1))
def vis_bilde(n=0):
    cap.set(cv2.CAP_PROP_POS_FRAMES, n)
    ok, bilde = cap.read()
    if ok:
        cv2.imshow("bilde", bilde)
        print(f"bilde {n}, {n / fps:.2f} s")

## Spill av og følg den røde ballen

Løkken leser ett bilde om gangen til videoen er slutt, akkurat som på PC-en. For hvert bilde:

1. Gjør bildet om til HSV, og lag en maske med `cv2.inRange` for rødt. Rødt ligger i begge ender av H (0–179), så det trengs to intervaller.
2. Finn den største hvite flekken i masken med `cv2.findContours`, og tegn en ring rundt den.
3. Tegn sporet ballen har fulgt så langt.

`cv2.imshow` med samme navn bytter ut bildet, så du ser videoen spilles av under cellen. `cv2.waitKey(33)` venter 33 ms mellom bildene, og det gir riktig fart: 1000 ms / 30 bilder per sekund ≈ 33 ms. Trykk **Stopp** for å avbryte.

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
spor = []
while True:
    ok, bilde = cap.read()
    if not ok:                      # ingen flere bilder: videoen er slutt
        break
    hsv = cv2.cvtColor(bilde, cv2.COLOR_BGR2HSV)
    maske = cv2.inRange(hsv, (0, 120, 70), (10, 255, 255)) | cv2.inRange(hsv, (170, 120, 70), (179, 255, 255))
    konturer, _ = cv2.findContours(maske, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if konturer:
        (x, y), r = cv2.minEnclosingCircle(max(konturer, key=cv2.contourArea))
        spor.append((int(x), int(y)))
        cv2.circle(bilde, (int(x), int(y)), int(r) + 5, (255, 255, 255), 2)
    for a, b in zip(spor, spor[1:]):
        cv2.line(bilde, a, b, (0, 255, 255), 2)
    cv2.imshow("sporing", bilde)
    if cv2.waitKey(round(1000 / fps)) & 0xFF == ord("q"):   # som på PC-en, men tastene kommer ikke fram her
        break

print(f"Fant den røde ballen i {len(spor)} av {antall} bilder.")

## Lagre resultatet som en ny video

`cv2.VideoWriter` skriver en video bilde for bilde. Her lagres kantene fra `cv2.Canny` som `kanter.mp4`. I nettleseren virker kodene

- `mp4v` til `.mp4`
- `MJPG` eller `XVID` til `.avi`

Nettleseren kan ikke spille av disse selv, men du kan laste ned filen eller legge den i `bilder/` fra **Filer** øverst til høyre, og spille den av i et videoprogram (for eksempel VLC). Du kan også lese den inn igjen med `cv2.VideoCapture`, som under.

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
ut = cv2.VideoWriter("kanter.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (bredde, hoyde))
print("VideoWriter er åpen:", ut.isOpened())
while True:
    ok, bilde = cap.read()
    if not ok:
        break
    kanter = cv2.Canny(cv2.cvtColor(bilde, cv2.COLOR_BGR2GRAY), 50, 150)
    ut.write(cv2.cvtColor(kanter, cv2.COLOR_GRAY2BGR))   # VideoWriter vil ha bilder med 3 kanaler
ut.release()

sjekk = cv2.VideoCapture("kanter.mp4")
print("kanter.mp4 har", int(sjekk.get(cv2.CAP_PROP_FRAME_COUNT)), "bilder")
sjekk.set(cv2.CAP_PROP_POS_FRAMES, 60)
cv2.imshow("kanter.mp4, bilde 60", sjekk.read()[1])

### Prøv selv

- Følg den grønne eller den blå ballen i stedet. I OpenCV er H gradene delt på 2, så grønn ligger rundt H = 60 og blå rundt H = 105–120. Vil du finne tallene selv, kan du lagre et bilde med `cv2.imwrite("bilde.png", bilde)`, trykke **Lagre i bilder/** i Filer-vinduet, og åpne det i Fargelab-fanen. Der viser markøren HSV-verdien til hver piksel.
- Finn det som beveger seg: `cv2.absdiff(forrige, bilde)` gir forskjellen mellom to bilder etter hverandre.
- Ta bort `cv2.waitKey` i løkken for å se hvor fort behandlingen går uten pauser.
- Bruk dine egne videoer: legg dem i mappen `bilder/`. Filer over 300 MB legges ikke i Python-mappen. AV1-video virker ikke. Kamera og IP-kamera finner du i `kamera_eksempel.ipynb`.

Til slutt kan du lukke videoen med `cap.release()`.